# Solar Panel Fault Detection - Image Classifier
## Deep Learning Internal Assessment 2 | Semester 6 | AI & Data Science

---

### 1. Problem Understanding

#### What Problem Are We Solving?
Solar panels are a critical renewable energy source, but their efficiency degrades significantly due to environmental and physical factors such as **dust accumulation, bird droppings, snow coverage, electrical faults, and physical damage**. Studies show that dust alone can reduce solar panel efficiency by **15-25%**, while physical damage or electrical faults can cause complete panel failure.

**Manual inspection** of large solar farms (with thousands of panels) is:
- Time-consuming and expensive
- Prone to human error
- Not scalable for remote installations

#### Why Does Automated Classification Matter?
An AI-based fault detection system can:
- **Reduce maintenance costs** by automatically identifying which panels need attention
- **Improve energy output** through early fault detection and targeted cleaning
- **Enable predictive maintenance** by monitoring panel health over time
- **Scale easily** to large solar farms using drone-captured imagery

#### Dataset Description
- **Source:** [Kaggle - Solar Panel Images](https://www.kaggle.com/datasets/pythonafroz/solar-panel-images)
- **6 Classes:**
  1. **Clean** (~193 images) - Normal, functioning panels
  2. **Dusty** (~190 images) - Dust-covered panels reducing light absorption
  3. **Bird-drop** (~198 images) - Bird droppings causing localized shading
  4. **Electrical-damage** (~103 images) - Hot spots, burnt cells, wiring issues
  5. **Physical-Damage** (~69 images) - Cracks, broken glass, structural damage
  6. **Snow-Covered** (~123 images) - Snow blocking sunlight completely
- **Total:** ~876 images
- **Class Imbalance:** Physical-Damage (69) vs Bird-drop (198) = nearly 3x difference

#### Key Challenges
1. **Class Imbalance:** Physical-Damage and Electrical-damage have significantly fewer samples
2. **Visual Similarity:** Dusty and Clean panels can look similar under certain lighting
3. **Varied Conditions:** Different lighting, camera angles, panel types, and image resolutions
4. **Small Dataset:** ~876 images is relatively small for deep learning - risk of overfitting

#### Real-World Relevance
This project directly addresses the growing need for **automated solar farm monitoring** as the world shifts toward renewable energy. With global solar capacity exceeding 1,000 GW, efficient fault detection is critical for maximizing return on investment.

In [1]:
# ============================================================
# Cell 1: Imports & Setup
# ============================================================
# All necessary libraries for the entire notebook.
# Setting random seeds at the top ensures full reproducibility.

import os
import sys
import ssl
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ---- Fix SSL certificates on macOS (needed for downloading pretrained weights) ----
if not os.environ.get('SSL_CERT_FILE'):
    try:
        import certifi
        os.environ['SSL_CERT_FILE'] = certifi.where()
    except ImportError:
        pass
ssl._create_default_https_context = ssl._create_unverified_context

# ---- Reproducibility: Set all random seeds ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ---- Add src directory to path so we can import our modules ----
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

# ---- Import our custom modules ----
from src.preprocess import (
    load_dataset, split_dataset, compute_class_weights,
    get_train_transforms, get_val_test_transforms, get_display_transforms,
    SolarPanelDataset, get_weighted_sampler, create_dataloaders,
    CLASS_NAMES, IMG_SIZE, BATCH_SIZE, IMAGENET_MEAN, IMAGENET_STD
)
from src.model import CustomCNN, EfficientNetTransfer, get_model_summary, count_parameters
from src.train import train_model, load_best_model
from src.evaluate import (
    evaluate_model, compare_models, get_predictions,
    compute_metrics, print_metrics, plot_confusion_matrix, get_classification_report
)

# ---- Device Configuration ----
# Use GPU if available for faster training, otherwise fall back to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# ---- Paths ----
# Try multiple possible dataset locations
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'Faulty_solar_panel'))
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Faulty_solar_panel'))
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'Faulty_solar_panel'))
    
SAVE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'outputs', 'models'))
PLOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'outputs', 'plots'))
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print(f'Dataset path: {DATA_DIR}')
print(f'Dataset exists: {os.path.exists(DATA_DIR)}')
print(f'Model save dir: {SAVE_DIR}')
print(f'Plot save dir: {PLOT_DIR}')

Using device: mps
PyTorch version: 2.2.2
Dataset path: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/data/Faulty_solar_panel
Dataset exists: True
Model save dir: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/models
Plot save dir: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots


---
## 2. Data Preprocessing & Augmentation

### Preprocessing Steps:
1. **Resize to 224x224** - Standard input size for CNN and transfer learning models (EfficientNet, ResNet). This ensures uniform input dimensions.
2. **Normalize with ImageNet mean/std** - Since we use EfficientNetB0 (pretrained on ImageNet), normalizing with ImageNet statistics ensures the input distribution matches what the pretrained model expects.
3. **Stratified Split (70/15/15)** - Maintains class proportions in train/val/test sets, critical when some classes have 3x fewer images.
4. **Class Weights + Weighted Sampling** - Dual approach to handle imbalance: class weights in loss function penalize minority class errors more; weighted sampler oversamples minority classes during training.

### Augmentation (Training Set Only):
| Augmentation | Justification |
|---|---|
| Random Horizontal Flip | Panels can appear from either direction; doesn't change fault type |
| Random Vertical Flip | Different camera orientations; adds rotational invariance |
| Random Rotation (20°) | Simulates slight camera angle variations in field captures |
| Random Resized Crop | Simulates zoom variation and different framing of panels |
| Color Jitter | Accounts for different lighting, weather, time of day |
| Random Affine | Simulates camera movement and varying distances |

In [2]:
# ============================================================
# Cell 2: Load Dataset & Explore Class Distribution
# ============================================================
# Load all image paths and labels, then visualize class distribution
# to understand the imbalance problem before training.

image_paths, labels, class_names = load_dataset(DATA_DIR)

# Count images per class
label_counts = Counter(labels)
class_counts = {CLASS_NAMES[k]: v for k, v in sorted(label_counts.items())}

print('\nClass Distribution:')
for name, count in class_counts.items():
    print(f'  {name}: {count} images')
print(f'  Total: {sum(class_counts.values())} images')

# ---- Plot class distribution bar chart ----
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#00BCD4']
bars = ax.bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black', linewidth=0.5)

# Add count labels on each bar
for bar, count in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Dataset Class Distribution (showing class imbalance)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fault Category', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution plot saved.')

Loaded 869 images across 6 classes

Class Distribution:
  Bird-drop: 191 images
  Clean: 193 images
  Dusty: 190 images
  Electrical-damage: 103 images
  Physical-Damage: 69 images
  Snow-Covered: 123 images
  Total: 869 images


Class distribution plot saved.


In [3]:
# ============================================================
# Cell 3: Create Data Splits & DataLoaders
# ============================================================
# Split dataset into train (70%), validation (15%), test (15%)
# using stratified splitting to maintain class proportions.
# Create DataLoaders with weighted sampling for the training set.

train_loader, val_loader, test_loader, class_weights, class_names = create_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=2
)

print(f'\nClass weights (for loss function):')
for name, w in zip(CLASS_NAMES, class_weights):
    print(f'  {name}: {w:.4f}')

Loaded 869 images across 6 classes

Dataset Split:
  Training:   607 images (70%)
  Validation: 131 images (15%)
  Test:       131 images (15%)
  Train distribution: {'Bird-drop': 133, 'Clean': 135, 'Dusty': 132, 'Electrical-damage': 72, 'Physical-Damage': 49, 'Snow-Covered': 86}
  Val distribution: {'Bird-drop': 29, 'Clean': 29, 'Dusty': 29, 'Electrical-damage': 16, 'Physical-Damage': 10, 'Snow-Covered': 18}
  Test distribution: {'Bird-drop': 29, 'Clean': 29, 'Dusty': 29, 'Electrical-damage': 15, 'Physical-Damage': 10, 'Snow-Covered': 19}

Class Weights (higher = more underrepresented):
  Bird-drop: 0.7607
  Clean: 0.7494
  Dusty: 0.7664
  Electrical-damage: 1.4051
  Physical-Damage: 2.0646
  Snow-Covered: 1.1764

DataLoaders created (batch_size=32):
  Train batches: 18
  Val batches:   5
  Test batches:  5

Class weights (for loss function):
  Bird-drop: 0.7607
  Clean: 0.7494
  Dusty: 0.7664
  Electrical-damage: 1.4051
  Physical-Damage: 2.0646
  Snow-Covered: 1.1764


In [4]:
# ============================================================
# Cell 4: Visualize Sample Augmented Images
# ============================================================
# Show original vs augmented images side-by-side to demonstrate
# how augmentation creates varied training samples from each image.

def show_augmented_samples(data_dir, class_names, num_samples=6):
    """Display original and augmented versions of sample images from each class."""
    train_tf = get_train_transforms()
    display_tf = get_display_transforms()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 3))
    fig.suptitle('Data Augmentation Examples (Original vs 3 Augmented Versions)', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    for i, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        # Get first valid image from the class
        valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
        img_files = [f for f in os.listdir(class_dir) 
                     if os.path.splitext(f)[1].lower() in valid_extensions]
        if not img_files:
            continue
        
        img_path = os.path.join(class_dir, img_files[0])
        img = Image.open(img_path).convert('RGB')
        
        # Original image (just resized, no augmentation)
        original = display_tf(img)
        axes[i, 0].imshow(original.permute(1, 2, 0).numpy())
        axes[i, 0].set_title(f'{class_name}\n(Original)', fontsize=10)
        axes[i, 0].axis('off')
        
        # 3 augmented versions of the same image
        for j in range(1, 4):
            augmented = train_tf(img)
            # Denormalize for display (reverse ImageNet normalization)
            mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
            std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
            augmented_display = augmented * std + mean
            augmented_display = torch.clamp(augmented_display, 0, 1)
            
            axes[i, j].imshow(augmented_display.permute(1, 2, 0).numpy())
            axes[i, j].set_title(f'Augmented v{j}', fontsize=10)
            axes[i, j].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'augmentation_samples.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Augmentation samples saved.')

show_augmented_samples(DATA_DIR, CLASS_NAMES)

Augmentation samples saved.


---
## 3. Model Architecture

We build and compare **two models**:

### Model A: Custom CNN (Built from Scratch)
- 4 Convolutional blocks with increasing filters (32 -> 64 -> 128 -> 256)
- Each block: Conv2D -> BatchNorm -> ReLU -> MaxPool2D
- Global Average Pooling -> FC(512) -> Dropout(0.5) -> FC(256) -> Dropout(0.5) -> Output(6)
- **Why:** Tests our ability to design a CNN architecture; provides baseline for comparison

### Model B: Transfer Learning with EfficientNetB0
- Pretrained on ImageNet (1.2M images, 1000 classes)
- Frozen base layers with custom classification head
- Fine-tuning of top 3 layers after initial training
- **Why:** Leverages pretrained features to achieve higher accuracy on our small dataset (~876 images)

In [5]:
# ============================================================
# Cell 5: Build Model A - Custom CNN
# ============================================================
# Initialize the Custom CNN and display its architecture summary.
# This model is built from scratch with 4 conv blocks.

model_a = CustomCNN(num_classes=6)
model_a = model_a.to(device)

# Print model summary with layer details
print('MODEL A: Custom CNN Architecture')
print('=' * 60)
get_model_summary(model_a, input_size=(3, 224, 224), device=str(device))

total_a, trainable_a = count_parameters(model_a)
print(f'\nTotal parameters: {total_a:,}')
print(f'Trainable parameters: {trainable_a:,}')

MODEL A: Custom CNN Architecture
torchsummary not installed. Printing basic model structure:
CustomCNN(
  (conv_block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, d

In [6]:
# ============================================================
# Cell 6: Build Model B - EfficientNetB0 Transfer Learning
# ============================================================
# Initialize EfficientNetB0 with pretrained ImageNet weights.
# Base layers are frozen initially (feature extraction mode).

model_b = EfficientNetTransfer(num_classes=6, pretrained=True)
model_b = model_b.to(device)

# Print model summary
print('MODEL B: EfficientNetB0 (Transfer Learning)')
print('=' * 60)
total_b, trainable_b = count_parameters(model_b)
print(f'Total parameters: {total_b:,}')
print(f'Trainable parameters: {trainable_b:,} (base frozen)')
print(f'Frozen parameters: {total_b - trainable_b:,}')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/shreyasgurav/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth



  0%|                                                                                                                                                                           | 0.00/20.5M [00:00<?, ?B/s]


  7%|███████████▋                                                                                                                                                      | 1.47M/20.5M [00:00<00:01, 15.0MB/s]


 17%|███████████████████████████▍                                                                                                                                      | 3.46M/20.5M [00:00<00:00, 18.3MB/s]


 26%|█████████████████████████████████████████▉                                                                                                                        | 5.30M/20.5M [00:00<00:00, 18.7MB/s]

...


 79%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                  | 16.1M/20.5M [00:00<00:00, 15.2MB/s]


 87%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                     | 17.8M/20.5M [00:01<00:00, 15.8MB/s]


 96%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎      | 19.6M/20.5M [00:01<00:00, 16.8MB/s]


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:01<00:00, 17.3MB/s]

Base model layers FROZEN (feature extraction mode)
MODEL B: EfficientNetB0 (Transfer Learning)
Total parameters: 4,337,026
Trainable parameters: 329,478 (base frozen)
Frozen parameters: 4,007,548


In [7]:
# ============================================================
# Cell 7: Generate Architecture Diagram
# ============================================================
# Create a visual diagram of both model architectures using matplotlib.
# This serves as the architecture_diagram.png for the report.

def draw_architecture_diagram(save_path):
    """Draw a detailed architecture diagram for both models."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 14))
    
    # ---- Model A: Custom CNN ----
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 16)
    ax1.set_title('Model A: Custom CNN', fontsize=16, fontweight='bold', pad=20)
    ax1.axis('off')
    
    layers_a = [
        ('Input\n3x224x224', '#E3F2FD', 15),
        ('Conv2D(32) + BN + ReLU\nMaxPool2D -> 32x112x112', '#BBDEFB', 13.5),
        ('Conv2D(64) + BN + ReLU\nMaxPool2D -> 64x56x56', '#90CAF9', 12),
        ('Conv2D(128) + BN + ReLU\nMaxPool2D -> 128x28x28', '#64B5F6', 10.5),
        ('Conv2D(256) + BN + ReLU\nMaxPool2D -> 256x14x14', '#42A5F5', 9),
        ('Global Avg Pool\n256x1x1', '#2196F3', 7.5),
        ('Flatten -> FC(512)\nReLU + Dropout(0.5)', '#1E88E5', 6),
        ('FC(256)\nReLU + Dropout(0.5)', '#1565C0', 4.5),
        ('FC(6) - Output\n6 classes (Softmax)', '#0D47A1', 3),
    ]
    
    for text, color, y in layers_a:
        rect = plt.Rectangle((1, y - 0.6), 8, 1.1, facecolor=color, 
                            edgecolor='black', linewidth=1.5, alpha=0.85)
        ax1.add_patch(rect)
        ax1.text(5, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Draw arrows
    for i in range(len(layers_a) - 1):
        y_start = layers_a[i][2] - 0.6
        y_end = layers_a[i+1][2] + 0.5
        ax1.annotate('', xy=(5, y_end), xytext=(5, y_start),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    # ---- Model B: EfficientNetB0 ----
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 16)
    ax2.set_title('Model B: EfficientNetB0 (Transfer Learning)', fontsize=16, fontweight='bold', pad=20)
    ax2.axis('off')
    
    layers_b = [
        ('Input\n3x224x224', '#FFF3E0', 15),
        ('EfficientNetB0 Backbone\n(Pretrained on ImageNet)\n[FROZEN initially]', '#FFE0B2', 13),
        ('Feature Maps\n1280 channels', '#FFCC80', 11),
        ('Global Avg Pool\n1280x1x1', '#FFB74D', 9.5),
        ('Dropout(0.4)', '#FFA726', 8),
        ('FC(256) + ReLU', '#FF9800', 6.5),
        ('Dropout(0.4)', '#FB8C00', 5),
        ('FC(6) - Output\n6 classes (Softmax)', '#E65100', 3.5),
    ]
    
    for text, color, y in layers_b:
        h = 1.5 if 'Backbone' in text else 1.1
        rect = plt.Rectangle((1, y - h/2 - 0.05), 8, h, facecolor=color,
                            edgecolor='black', linewidth=1.5, alpha=0.85)
        ax2.add_patch(rect)
        ax2.text(5, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    
    for i in range(len(layers_b) - 1):
        y_start = layers_b[i][2] - 0.8
        y_end = layers_b[i+1][2] + 0.6
        ax2.annotate('', xy=(5, y_end), xytext=(5, y_start),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    # Fine-tuning annotation
    ax2.text(5, 1.5, 'Phase 1: Train classifier head (base frozen)\nPhase 2: Unfreeze top 3 blocks + fine-tune',
             ha='center', va='center', fontsize=10, style='italic',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='orange', alpha=0.9))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Architecture diagram saved to: {save_path}')

arch_diagram_path = os.path.join(os.getcwd(), '..', 'architecture_diagram.png')
draw_architecture_diagram(arch_diagram_path)

Architecture diagram saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/notebooks/../architecture_diagram.png


---
## 4. Training

### Training Configuration:
| Parameter | Value | Justification |
|---|---|---|
| Loss Function | CrossEntropyLoss (weighted) | Handles multi-class classification; class weights address imbalance |
| Optimizer | Adam (lr=0.001) | Adaptive learning rate; works well for most deep learning tasks |
| L2 Regularization | weight_decay=1e-4 | Prevents overfitting by penalizing large weights |
| LR Scheduler | ReduceLROnPlateau | Reduces LR when validation loss plateaus, helping escape local minima |
| Early Stopping | patience=5 | Stops training when model stops improving, prevents overfitting |
| Batch Size | 32 | Good balance between gradient stability and memory usage |
| Epochs | 25 (max) | With early stopping, actual epochs may be fewer |
| Dropout | 0.5 (CNN) / 0.4 (EfficientNet) | Regularization to prevent overfitting on small dataset |

In [8]:
# ============================================================
# Cell 8: Train Model A - Custom CNN
# ============================================================
# Train the custom CNN from scratch with all regularization techniques.
# Training includes: weighted loss, Adam + L2, LR scheduling, early stopping.

print('Training Model A: Custom CNN')
print('=' * 60)

# Re-initialize model to ensure fresh weights
model_a = CustomCNN(num_classes=6)

history_a = train_model(
    model=model_a,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    device=device,
    model_name='custom_cnn',
    num_epochs=25,
    learning_rate=0.001,
    weight_decay=1e-4,
    patience=5,
    save_dir=SAVE_DIR,
    plot_dir=PLOT_DIR
)

Training Model A: Custom CNN

Training custom_cnn
Device: mps
Epochs: 25 (early stopping patience: 5)
Learning Rate: 0.001
Weight Decay (L2): 0.0001

Epoch [1/25] (LR: 0.001000)



Training:   0%|                                                                                                                                                                      | 0/18 [00:00<?, ?it/s]


Training:   0%|                                                                                                                                                         | 0/18 [00:09<?, ?it/s, loss=1.8080]


Training:   6%|████████                                                                                                                                         | 1/18 [00:09<02:47,  9.83s/it, loss=1.8080]


Training:   6%|████████                                                                                                                                         | 1/18 [00:09<02:47,  9.83s/it, loss=1.8205]

...


Validating:  20%|█████                    | 1/5 [00:02<00:11,  2.87s/it]


Validating:  60%|███████████████          | 3/5 [00:03<00:01,  1.24it/s]


Validating:  80%|████████████████████     | 4/5 [00:03<00:00,  1.75it/s]

  Train Loss: 0.9955 | Train Acc: 56.94%
  Val Loss:   1.1205 | Val Acc:   60.31%
  No improvement for 5/5 epochs

Early stopping triggered after 23 epochs!
Best Val Acc: 64.89% at epoch with Val Loss: 1.1800


Loss curve saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/custom_cnn_loss_curve.png
Accuracy curve saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/custom_cnn_accuracy_curve.png

Training complete for custom_cnn!
Best Validation Accuracy: 64.89%
Model saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/models/custom_cnn_best.pth


In [9]:
# ============================================================
# Cell 9: Train Model B - EfficientNetB0 (Phase 1: Feature Extraction)
# ============================================================
# Phase 1: Train only the custom classifier head while keeping
# EfficientNet backbone frozen. This learns good classifier weights
# without destroying pretrained features.

print('Training Model B: EfficientNetB0 - Phase 1 (Feature Extraction)')
print('=' * 60)

# Re-initialize model with frozen base
model_b = EfficientNetTransfer(num_classes=6, pretrained=True)

history_b_phase1 = train_model(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    device=device,
    model_name='efficientnet_phase1',
    num_epochs=10,
    learning_rate=0.001,
    weight_decay=1e-4,
    patience=5,
    save_dir=SAVE_DIR,
    plot_dir=PLOT_DIR
)

Training Model B: EfficientNetB0 - Phase 1 (Feature Extraction)


Base model layers FROZEN (feature extraction mode)

Training efficientnet_phase1
Device: mps
Epochs: 10 (early stopping patience: 5)
Learning Rate: 0.001
Weight Decay (L2): 0.0001

Epoch [1/10] (LR: 0.001000)



Training:   0%|                                  | 0/18 [00:00<?, ?it/s]


Training:   0%|                     | 0/18 [00:05<?, ?it/s, loss=1.8238]


Training:   6%|▋            | 1/18 [00:05<01:41,  5.99s/it, loss=1.8238]

...


Validating:   0%|                                 | 0/5 [00:00<?, ?it/s]


Validating:  20%|█████                    | 1/5 [00:03<00:13,  3.39s/it]


Validating:  60%|███████████████          | 3/5 [00:03<00:01,  1.09it/s]

  Train Loss: 0.6271 | Train Acc: 71.53%
  Val Loss:   0.7426 | Val Acc:   73.28%
  ✓ Best model saved (Val Acc: 73.28%)



Loss curve saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/efficientnet_phase1_loss_curve.png
Accuracy curve saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/efficientnet_phase1_accuracy_curve.png

Training complete for efficientnet_phase1!
Best Validation Accuracy: 73.28%
Model saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/models/efficientnet_phase1_best.pth


In [10]:
# ============================================================
# Cell 10: Train Model B - Phase 2 (Fine-Tuning)
# ============================================================
# Phase 2: Unfreeze the last 3 blocks of EfficientNet and continue
# training with a lower learning rate. This adapts the high-level
# features to our specific solar panel classification task.

print('Training Model B: EfficientNetB0 - Phase 2 (Fine-Tuning)')
print('=' * 60)

# Unfreeze top 3 layers for fine-tuning
model_b.unfreeze_top_layers(num_layers=3)

total_b, trainable_b = count_parameters(model_b)
print(f'Trainable parameters after unfreezing: {trainable_b:,}')

# Fine-tune with a lower learning rate to avoid destroying pretrained features
history_b_phase2 = train_model(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    device=device,
    model_name='efficientnet_finetuned',
    num_epochs=15,
    learning_rate=0.0001,  # 10x lower LR for fine-tuning
    weight_decay=1e-4,
    patience=5,
    save_dir=SAVE_DIR,
    plot_dir=PLOT_DIR
)

Training Model B: EfficientNetB0 - Phase 2 (Fine-Tuning)
Unfroze last 3 blocks (fine-tuning mode)
Trainable parameters after unfreezing: 3,485,218

Training efficientnet_finetuned
Device: mps
Epochs: 15 (early stopping patience: 5)
Learning Rate: 0.0001
Weight Decay (L2): 0.0001

Epoch [1/15] (LR: 0.000100)



Training:   0%|                                  | 0/18 [00:00<?, ?it/s]


Training:   0%|                     | 0/18 [00:06<?, ?it/s, loss=0.5567]


Training:   6%|▋            | 1/18 [00:06<01:56,  6.87s/it, loss=0.5567]


Training:   6%|▋            | 1/18 [00:06<01:56,  6.87s/it, loss=0.5202]

...


Validating:  20%|█████                    | 1/5 [00:03<00:12,  3.00s/it]


Validating:  60%|███████████████          | 3/5 [00:03<00:01,  1.22it/s]


Validating: 100%|█████████████████████████| 5/5 [00:03<00:00,  2.29it/s]

  Train Loss: 0.1751 | Train Acc: 92.53%
  Val Loss:   0.4716 | Val Acc:   83.97%
  No improvement for 3/5 epochs



Loss curve saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/efficientnet_finetuned_loss_curve.png
Accuracy curve saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/efficientnet_finetuned_accuracy_curve.png

Training complete for efficientnet_finetuned!
Best Validation Accuracy: 84.73%
Model saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/models/efficientnet_finetuned_best.pth


In [11]:
# ============================================================
# Cell 11: Combined Training Curves for EfficientNet
# ============================================================
# Combine Phase 1 and Phase 2 training histories for a complete view.

# Merge histories
history_b_combined = {
    'train_loss': history_b_phase1['train_loss'] + history_b_phase2['train_loss'],
    'val_loss': history_b_phase1['val_loss'] + history_b_phase2['val_loss'],
    'train_acc': history_b_phase1['train_acc'] + history_b_phase2['train_acc'],
    'val_acc': history_b_phase1['val_acc'] + history_b_phase2['val_acc'],
}

epochs = range(1, len(history_b_combined['train_loss']) + 1)
phase1_end = len(history_b_phase1['train_loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Loss curve
ax1.plot(epochs, history_b_combined['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax1.plot(epochs, history_b_combined['val_loss'], 'r-o', label='Val Loss', markersize=4)
ax1.axvline(x=phase1_end, color='green', linestyle='--', label=f'Fine-tuning starts (epoch {phase1_end})')
ax1.set_title('EfficientNetB0 - Loss Curve (Phase 1 + 2)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(epochs, history_b_combined['train_acc'], 'b-o', label='Train Acc', markersize=4)
ax2.plot(epochs, history_b_combined['val_acc'], 'r-o', label='Val Acc', markersize=4)
ax2.axvline(x=phase1_end, color='green', linestyle='--', label=f'Fine-tuning starts (epoch {phase1_end})')
ax2.set_title('EfficientNetB0 - Accuracy Curve (Phase 1 + 2)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'efficientnet_combined_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Combined training curves saved.')

Combined training curves saved.


---
## 5. Evaluation & Metrics

We evaluate both models on the **held-out test set** (15% of data, never seen during training) and report:

| Metric | What It Measures |
|---|---|
| **Accuracy** | Overall percentage of correct predictions |
| **Precision** | Of all predicted as class X, how many were actually class X (measures false positives) |
| **Recall** | Of all actual class X, how many were correctly predicted (measures false negatives) |
| **F1-Score** | Harmonic mean of Precision and Recall (balanced metric) |
| **Confusion Matrix** | Shows exactly which classes are being confused with each other |

We use **macro averaging** to give equal weight to all classes regardless of their size, which is important for our imbalanced dataset.

In [12]:
# ============================================================
# Cell 12: Evaluate Model A - Custom CNN
# ============================================================
# Load the best saved model weights and evaluate on the test set.

print('Evaluating Model A: Custom CNN')
print('=' * 60)

# Load best model weights
model_a_eval = CustomCNN(num_classes=6)
best_model_a_path = os.path.join(SAVE_DIR, 'custom_cnn_best.pth')
model_a_eval = load_best_model(model_a_eval, best_model_a_path, device)

# Full evaluation
metrics_a, preds_a, labels_a = evaluate_model(
    model=model_a_eval,
    test_loader=test_loader,
    class_names=CLASS_NAMES,
    device=device,
    model_name='Custom_CNN',
    save_dir=PLOT_DIR
)

Evaluating Model A: Custom CNN
Loaded best model from: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/models/custom_cnn_best.pth
  Checkpoint epoch: 18, Val Acc: 64.89%

Evaluating Custom_CNN on test set...



Evaluating:   0%|                                 | 0/5 [00:00<?, ?it/s]


Evaluating:  20%|█████                    | 1/5 [00:04<00:16,  4.14s/it]


Evaluating:  80%|████████████████████     | 4/5 [00:04<00:00,  1.16it/s]


  Evaluation Results: Custom_CNN

  Overall Accuracy: 54.96%

  Class                   Precision     Recall   F1-Score
  ----------------------------------------------------
  Bird-drop                  83.33%     17.24%     28.57%
  Clean                      56.25%     62.07%     59.02%
  Dusty                      45.45%     68.97%     54.79%
  Electrical-damage          54.17%     86.67%     66.67%
  Physical-Damage            40.00%     40.00%     40.00%
  Snow-Covered               80.00%     63.16%     70.59%
  ----------------------------------------------------
  Macro Average              59.87%     56.35%     53.27%
  Weighted Average           61.82%     54.96%     52.44%



Confusion matrix saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/Custom_CNN_confusion_matrix.png
Classification Report:
                   precision    recall  f1-score   support

        Bird-drop     0.8333    0.1724    0.2857        29
            Clean     0.5625    0.6207    0.5902        29
            Dusty     0.4545    0.6897    0.5479        29
Electrical-damage     0.5417    0.8667    0.6667        15
  Physical-Damage     0.4000    0.4000    0.4000        10
     Snow-Covered     0.8000    0.6316    0.7059        19

         accuracy                         0.5496       131
        macro avg     0.5987    0.5635    0.5327       131
     weighted avg     0.6182    0.5496    0.5244       131



In [13]:
# ============================================================
# Cell 13: Evaluate Model B - EfficientNetB0
# ============================================================
# Load the best fine-tuned EfficientNet model and evaluate.

print('Evaluating Model B: EfficientNetB0 (Fine-tuned)')
print('=' * 60)

# Load best fine-tuned model weights
model_b_eval = EfficientNetTransfer(num_classes=6, pretrained=False)
best_model_b_path = os.path.join(SAVE_DIR, 'efficientnet_finetuned_best.pth')
model_b_eval = load_best_model(model_b_eval, best_model_b_path, device)

# Full evaluation
metrics_b, preds_b, labels_b = evaluate_model(
    model=model_b_eval,
    test_loader=test_loader,
    class_names=CLASS_NAMES,
    device=device,
    model_name='EfficientNetB0',
    save_dir=PLOT_DIR
)

Evaluating Model B: EfficientNetB0 (Fine-tuned)
Base model layers FROZEN (feature extraction mode)


Loaded best model from: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/models/efficientnet_finetuned_best.pth
  Checkpoint epoch: 12, Val Acc: 84.73%

Evaluating EfficientNetB0 on test set...



Evaluating:   0%|                                 | 0/5 [00:00<?, ?it/s]


Evaluating:  20%|█████                    | 1/5 [00:02<00:11,  2.77s/it]


Evaluating:  60%|███████████████          | 3/5 [00:03<00:01,  1.20it/s]


Evaluating:  80%|████████████████████     | 4/5 [00:03<00:00,  1.44it/s]


  Evaluation Results: EfficientNetB0

  Overall Accuracy: 84.73%

  Class                   Precision     Recall   F1-Score
  ----------------------------------------------------
  Bird-drop                  85.71%     82.76%     84.21%
  Clean                      78.79%     89.66%     83.87%
  Dusty                      82.61%     65.52%     73.08%
  Electrical-damage          83.33%    100.00%     90.91%
  Physical-Damage            80.00%     80.00%     80.00%
  Snow-Covered              100.00%    100.00%    100.00%
  ----------------------------------------------------
  Macro Average              85.07%     86.32%     85.34%
  Weighted Average           84.86%     84.73%     84.41%



Confusion matrix saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/EfficientNetB0_confusion_matrix.png
Classification Report:
                   precision    recall  f1-score   support

        Bird-drop     0.8571    0.8276    0.8421        29
            Clean     0.7879    0.8966    0.8387        29
            Dusty     0.8261    0.6552    0.7308        29
Electrical-damage     0.8333    1.0000    0.9091        15
  Physical-Damage     0.8000    0.8000    0.8000        10
     Snow-Covered     1.0000    1.0000    1.0000        19

         accuracy                         0.8473       131
        macro avg     0.8507    0.8632    0.8534       131
     weighted avg     0.8486    0.8473    0.8441       131



In [14]:
# ============================================================
# Cell 14: Model Comparison
# ============================================================
# Side-by-side comparison of both models on all metrics.

compare_models(
    metrics_a=metrics_a,
    metrics_b=metrics_b,
    model_name_a='Custom CNN',
    model_name_b='EfficientNetB0',
    save_dir=PLOT_DIR
)

Model comparison plot saved to: /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots/model_comparison.png

  Model Comparison Summary
  Metric                    Custom CNN  EfficientNetB0
  --------------------------------------------------
  Accuracy                      54.96%          84.73% →
  Precision (Macro)             59.87%          85.07% →
  Recall (Macro)                56.35%          86.32% →
  F1-Score (Macro)              53.27%          85.34% →



In [15]:
# ============================================================
# Cell 15: Display Confusion Matrices Side by Side
# ============================================================

from sklearn.metrics import confusion_matrix

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Custom CNN confusion matrix
cm_a = confusion_matrix(labels_a, preds_a)
sns.heatmap(cm_a, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, square=True, linewidths=0.5, ax=ax1)
ax1.set_title('Custom CNN - Confusion Matrix', fontsize=13, fontweight='bold')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')

# EfficientNet confusion matrix
cm_b = confusion_matrix(labels_b, preds_b)
sns.heatmap(cm_b, annot=True, fmt='d', cmap='Oranges', xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, square=True, linewidths=0.5, ax=ax2)
ax2.set_title('EfficientNetB0 - Confusion Matrix', fontsize=13, fontweight='bold')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrix_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Side-by-side confusion matrices saved.')

Side-by-side confusion matrices saved.


In [16]:
# ============================================================
# Cell 16: Summary Results Table
# ============================================================
# Create a clean comparison table using pandas for the report.

import pandas as pd

results_data = {
    'Metric': ['Accuracy (%)', 'Precision - Macro (%)', 'Recall - Macro (%)', 
               'F1-Score - Macro (%)', 'Precision - Weighted (%)', 
               'Recall - Weighted (%)', 'F1-Score - Weighted (%)'],
    'Custom CNN': [
        f"{metrics_a['accuracy']:.2f}",
        f"{metrics_a['precision_macro']:.2f}",
        f"{metrics_a['recall_macro']:.2f}",
        f"{metrics_a['f1_macro']:.2f}",
        f"{metrics_a['precision_weighted']:.2f}",
        f"{metrics_a['recall_weighted']:.2f}",
        f"{metrics_a['f1_weighted']:.2f}",
    ],
    'EfficientNetB0': [
        f"{metrics_b['accuracy']:.2f}",
        f"{metrics_b['precision_macro']:.2f}",
        f"{metrics_b['recall_macro']:.2f}",
        f"{metrics_b['f1_macro']:.2f}",
        f"{metrics_b['precision_weighted']:.2f}",
        f"{metrics_b['recall_weighted']:.2f}",
        f"{metrics_b['f1_weighted']:.2f}",
    ]
}

results_df = pd.DataFrame(results_data)
print('\nResults Summary Table:')
print(results_df.to_string(index=False))

# Save as CSV too
results_df.to_csv(os.path.join(PLOT_DIR, 'results_summary.csv'), index=False)
print('\nResults saved to outputs/plots/results_summary.csv')


Results Summary Table:
                  Metric Custom CNN EfficientNetB0
            Accuracy (%)      54.96          84.73
   Precision - Macro (%)      59.87          85.07
      Recall - Macro (%)      56.35          86.32
    F1-Score - Macro (%)      53.27          85.34
Precision - Weighted (%)      61.82          84.86
   Recall - Weighted (%)      54.96          84.73
 F1-Score - Weighted (%)      52.44          84.41

Results saved to outputs/plots/results_summary.csv


---
## 6. Results Interpretation & Conclusion

### Which Model Performed Better?

**EfficientNetB0 (Transfer Learning) significantly outperformed the Custom CNN**, achieving **84.73% accuracy** compared to the Custom CNN's **54.96%** — a difference of nearly 30 percentage points. This was consistent across all metrics:

| Metric | Custom CNN | EfficientNetB0 | Improvement |
|---|---|---|---|
| Accuracy | 54.96% | **84.73%** | +29.77% |
| Precision (Macro) | 59.87% | **85.07%** | +25.20% |
| Recall (Macro) | 56.35% | **86.32%** | +29.97% |
| F1-Score (Macro) | 53.27% | **85.34%** | +32.07% |

### Why Did EfficientNetB0 Win?

1. **Pretrained Features:** EfficientNetB0 was trained on 1.2 million ImageNet images, giving it rich, general-purpose visual features (edges, textures, shapes) that transferred directly to solar panel fault detection. The Custom CNN had to learn all features from scratch with only ~607 training images.

2. **Data Efficiency:** With only ~607 training images, the Custom CNN struggled to learn robust features and was prone to overfitting. EfficientNet's pretrained knowledge effectively expanded the training data by leveraging patterns learned from millions of images.

3. **Architecture Superiority:** EfficientNet uses compound scaling (depth, width, resolution) optimized through neural architecture search, making it fundamentally more efficient than our hand-designed 4-block CNN.

4. **Two-Phase Training:** The fine-tuning strategy (Phase 1: frozen backbone → Phase 2: unfreeze top layers with lower LR) allowed the model to first learn a good classifier head, then carefully adapt high-level features to our domain without destroying pretrained knowledge.

### Confusion Matrix Analysis

From the confusion matrices:
- **EfficientNetB0** classified most classes with high accuracy, with the strongest performance on **Snow-Covered** (19/19 = 100%) and **Clean** (26/29 = 89.7%).
- **Custom CNN** struggled most with **Bird-drop** (only 5/29 correct) and **Physical-Damage** (4/10 correct), frequently confusing them with Dusty panels.
- Both models found **Dusty** challenging — it was most commonly confused with Bird-drop and Clean, which makes sense as light dust can visually resemble a clean panel.
- **Physical-Damage** had the fewest training samples (~49), making it the hardest class for both models.

### Limitations

1. **Small Dataset:** ~869 total images is relatively small for deep learning. Models may not generalize well to significantly different solar panel types or environments.
2. **Class Imbalance:** Despite using class weights and weighted sampling, Physical-Damage (69 images) remains challenging due to limited examples.
3. **Visual Similarity:** Some fault types (dusty vs clean, physical vs electrical damage) have subtle visual differences that even human experts may find challenging.
4. **Data Quality:** Images are scraped from the internet with varying quality, resolution, lighting, and camera angles.
5. **Single Label:** Each image has exactly one label, but real panels could have multiple concurrent faults.

### Future Improvements

1. **More Data:** Collect more images, especially for underrepresented classes (Physical-Damage, Electrical-damage).
2. **GAN-based Augmentation:** Use Generative Adversarial Networks to synthesize realistic minority class images.
3. **Ensemble Models:** Combine predictions from multiple models (CNN + EfficientNet + ResNet) for higher accuracy.
4. **Object Detection:** Instead of whole-image classification, use YOLO or Faster R-CNN to localize and classify faults within the panel.
5. **Multi-label Classification:** Allow models to detect multiple simultaneous faults.
6. **Attention Mechanisms:** Add attention layers to focus on fault-specific regions of the panel.
7. **Cross-validation:** Use k-fold cross-validation for more robust performance estimates.

### Overall Conclusion

This project demonstrated the effectiveness of deep learning for **automated solar panel fault detection**. Transfer learning with EfficientNetB0 achieved **84.73% accuracy** and **85.34% macro F1-score** across all 6 fault categories, significantly outperforming the Custom CNN (54.96% accuracy). The results confirm that **transfer learning is essential** when working with limited data (~869 images), and proper handling of class imbalance (through weighted loss + oversampling) is critical for achieving balanced performance across all fault types. The system can be integrated into drone-based inspection pipelines to enable **real-time, scalable monitoring** of solar farms.

In [17]:
# ============================================================
# Cell 17: Sample Predictions Visualization
# ============================================================
# Show some test images with their true vs predicted labels
# to give a visual sense of model performance.

def show_sample_predictions(model, test_loader, class_names, device, model_name, num_samples=12):
    """Display sample test images with true and predicted labels."""
    model.eval()
    images_shown = 0
    all_images = []
    all_true = []
    all_pred = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images_dev = images.to(device)
            outputs = model(images_dev)
            _, predicted = torch.max(outputs, 1)
            
            for i in range(images.size(0)):
                if images_shown >= num_samples:
                    break
                all_images.append(images[i])
                all_true.append(labels[i].item())
                all_pred.append(predicted[i].item())
                images_shown += 1
            if images_shown >= num_samples:
                break
    
    # Plot
    cols = 4
    rows = (num_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    fig.suptitle(f'Sample Predictions - {model_name}', fontsize=16, fontweight='bold')
    
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    
    for idx in range(num_samples):
        row, col = idx // cols, idx % cols
        ax = axes[row, col] if rows > 1 else axes[col]
        
        # Denormalize image
        img = all_images[idx] * std + mean
        img = torch.clamp(img, 0, 1)
        ax.imshow(img.permute(1, 2, 0).numpy())
        
        true_label = class_names[all_true[idx]]
        pred_label = class_names[all_pred[idx]]
        color = 'green' if all_true[idx] == all_pred[idx] else 'red'
        ax.set_title(f'True: {true_label}\nPred: {pred_label}', 
                     fontsize=9, color=color, fontweight='bold')
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(num_samples, rows * cols):
        row, col = idx // cols, idx % cols
        ax = axes[row, col] if rows > 1 else axes[col]
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'{model_name}_sample_predictions.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('Sample predictions from EfficientNetB0:')
show_sample_predictions(model_b_eval, test_loader, CLASS_NAMES, device, 'EfficientNetB0', num_samples=12)
print('\nSample predictions from Custom CNN:')
show_sample_predictions(model_a_eval, test_loader, CLASS_NAMES, device, 'Custom_CNN', num_samples=12)

Sample predictions from EfficientNetB0:



Sample predictions from Custom CNN:


In [18]:
# ============================================================
# Cell 18: Final Summary
# ============================================================

print('=' * 60)
print('  SOLAR PANEL FAULT DETECTION - PROJECT COMPLETE')
print('=' * 60)
print(f'\nModels trained and evaluated successfully!')
print(f'\nSaved outputs:')
print(f'  Models:  {SAVE_DIR}')
print(f'  Plots:   {PLOT_DIR}')
print(f'\nFiles generated:')

for root, dirs, files in os.walk(os.path.join(os.getcwd(), '..', 'outputs')):
    for f in sorted(files):
        rel_path = os.path.relpath(os.path.join(root, f), os.path.join(os.getcwd(), '..'))
        print(f'  - {rel_path}')

print(f'\nProject completed successfully!')

  SOLAR PANEL FAULT DETECTION - PROJECT COMPLETE

Models trained and evaluated successfully!

Saved outputs:
  Models:  /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/models
  Plots:   /Users/shreyasgurav/Desktop/DL IA2/solar_panel_classifier/outputs/plots

Files generated:
  - outputs/models/custom_cnn_best.pth
  - outputs/models/efficientnet_finetuned_best.pth
  - outputs/models/efficientnet_phase1_best.pth
  - outputs/plots/Custom_CNN_confusion_matrix.png
  - outputs/plots/Custom_CNN_sample_predictions.png
  - outputs/plots/EfficientNetB0_confusion_matrix.png
  - outputs/plots/EfficientNetB0_sample_predictions.png
  - outputs/plots/augmentation_samples.png
  - outputs/plots/class_distribution.png
  - outputs/plots/confusion_matrix_comparison.png
  - outputs/plots/custom_cnn_accuracy_curve.png
  - outputs/plots/custom_cnn_loss_curve.png
  - outputs/plots/efficientnet_combined_curves.png
  - outputs/plots/efficientnet_finetuned_accuracy_curve.png
  - outputs/plots/e